In [ ]:
from wrapt import decorator
from functools import wraps, lru_cache

In [ ]:
def defattr(object, attrname, default):
    if not hasattr(object, attrname):
        setattr(object, attrname, default)
    return getattr(object, attrname)

In [ ]:
def dispatch(function):
    instances = defattr(dispatch, "__instances__", {})
    instances.setdefault(function.__name__, {})
    input_types = tuple(list(function.__annotations__.values())[:-1])
    instances[function.__name__][input_types] = function
    
    @wraps(function)
    def wrapped(*args, **kwargs):
        input_types = tuple(type(arg) for arg in args)
        if function.__name__ not in instances:
            raise ValueError(f"Couldn't find a dispatcher for function {function.__name__}."
                             f"Known functions are {[name for name in instances.keys()]}")
        if input_types not in instances[function.__name__]:
            raise ValueError(f"Couldn't find an instance for input types {input_types}. Known types are {instances}.")
        return instances[function.__name__][input_types](*args)
    return wrapped

In [ ]:
_TYPECLASS_BASES = {}
_TYPECLASS_BINDINGS = {}
_TYPECLASS_NAMES = {}

def does_binding_exists(typeclass, instance):
    return (typeclass in _TYPECLASS_BINDINGS and 
            instance in _TYPECLASS_BINDINGS[typeclass])

class _SpecificTypeclass:
    
    def __init__(self, typeclass, instance):
        self.typeclass = typeclass
        self.instance = instance
        
    @lru_cache(maxsize=None)
    def __getattr__(self, attr_name):
        if does_binding_exists(self.typeclass, self.instance):
            return _TYPECLASS_BINDINGS[self.typeclass][self.instance]
        else:
            method = getattr(self.typeclass.type_instance, attr_name)
            return lambda *args, **kwargs: f(self, *args, **kwargs)
        
    def __repr__(self):
        return f"<_SpecificTypeclass {self.typeclass}[{self.instance}]>"
    
    
class Typeclass:
    def __init__(self, type_instance):
        self.type_instance = type_instance

    @lru_cache(maxsize=None)
    def __getitem__(self, instance):
        if isinstance(instance, self.type_instance):
            return instance
        else:
            return _SpecificTypeclass(self, instance)
        
    def __repr__(self):
        return f"<Typeclass {self.type_instance.__name__}>"


def implement_single(tc, cls, /, *, name=None):
    def decorator(function):
        if cls not in _TYPECLASS_BINDINGS[tc]:
            _TYPECLASS_BINDINGS[tc][cls] = {}
        _TYPECLASS_BINDINGS[tc][cls][name or function.__name__] = function
        return function
    return decorator
 
 
def instance(target_type):
    def decorator(cls):
        tc = _TYPECLASS_NAMES[cls.__name__]
        for attr in dir(cls):
            if not attr.startswith("__"):
                function = getattr(cls, attr)
                implement_single(tc, target_type, name=attr)(function)
        return tc
    return decorator
 
 
def typeclass(cls):
    tc = Typeclass(cls)
    _TYPECLASS_BINDINGS[tc] = {}
    _TYPECLASS_BASES[cls] = tc
    _TYPECLASS_NAMES[cls.__name__] = tc
    return tc
